# Lemonade + llama.cpp + Qwen3.6-35B-A3B-GGUF on AMD Radeon (single GPU)

This notebook runs **inside the `lemonadeon` container** and deploys **Lemonade Server** with the
**ROCm llama.cpp** backend, serving the `Qwen3.6-35B-A3B` MoE model on a single
**AMD Radeon GPU**.

Unlike the local-model version, here **Lemonade downloads the GGUF at runtime** from a Hugging Face
mirror into the workspace cache (`/workspace/hf_cache`), then serves it — a "download to local, then
use" flow. No pre-existing model file is required.

**Steps**

1. Configure the HF model, GPU, and runtime parameters.
2. Prepare Lemonade: install the SDK, reuse the prebuilt ROCm `llama-server`, and register the model's HF checkpoint.
3. Start the OpenAI-compatible Lemonade Server.
4. **Download** the model with Lemonade `pull` (~22 GB on first run, then cached).
5. Load the model and send a smoke-test request.
6. (Optional) Measure idle vs. peak VRAM.
7. Stop the server.

OpenAI-compatible endpoint: `http://127.0.0.1:13305/api/v1`

## 1. Configuration

The model is specified as a Hugging Face `repo:variant` checkpoint and downloaded at runtime.
`MODEL_VARIANT` is a quantization token; Lemonade resolves it to a single file in the repo
(`Q4_K_XL` → `Qwen3.6-35B-A3B-UD-Q4_K_XL.gguf`). Downloads go through the mirror in `HF_ENDPOINT`
into `HF_HOME`. `LLAMA_SERVER_BIN` is the prebuilt ROCm `llama-server` (build 9766 with
`libggml-hip.so`) shipped in the base image.

In [ ]:
import os
import sys
from pathlib import Path

# --- User-configurable settings (paths are inside the lemonadeon container) ---
WORKSPACE = Path("/workspace")

# Model is downloaded at runtime by Lemonade from the Hugging Face mirror.
MODEL_REPO = "unsloth/Qwen3.6-35B-A3B-GGUF"   # HF repo
MODEL_VARIANT = "Q4_K_XL"                       # quant token -> Qwen3.6-35B-A3B-UD-Q4_K_XL.gguf
MODEL_CHECKPOINT = f"{MODEL_REPO}:{MODEL_VARIANT}"
MODEL_NAME = "Qwen3.6-35B-A3B-GGUF"             # registered name
SERVED_MODEL = f"user.{MODEL_NAME}"             # OpenAI-compatible model id

GPU_ID = "0"                                    # single-GPU target
HOST = "0.0.0.0"
PORT = 13305
CTX_SIZE = 8192

# Prebuilt ROCm llama.cpp shipped in the base image (build 9766 + libggml-hip)
LLAMA_CPP_DIR = Path("/opt/llama.cpp")
LLAMA_SERVER_BIN = LLAMA_CPP_DIR / "build/bin/llama-server"
LLAMA_VERSION_ROCM = "b1066"

# Hugging Face download settings (mirror + cache inside the workspace)
HF_ENDPOINT = "http://134.199.133.77"
HF_HOME = WORKSPACE / "hf_cache"
os.environ["HF_ENDPOINT"] = HF_ENDPOINT
os.environ["HF_HOME"] = str(HF_HOME)
# NOTE: this image has hf_xet removed from the lemonade venv, so huggingface_hub
# downloads via the mirror above (Xet would otherwise hit huggingface.co).
# NOTE: This image already has the `hf_xet` package removed from the lemonade venv,
# so huggingface_hub falls back to regular HTTP downloads via the mirror above.
# (The Xet protocol would otherwise connect directly to huggingface.co, which is
# the fix is that hf_xet is not installed.)
HF_HOME.mkdir(parents=True, exist_ok=True)

LEMONADE_CACHE = WORKSPACE / "lemonade/cache"
LOG_DIR = WORKSPACE / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
SERVER_LOG = LOG_DIR / "lemonade_server.log"

API_BASE = f"http://127.0.0.1:{PORT}/api/v1"

print("Model checkpoint  :", MODEL_CHECKPOINT)
print("Served model      :", SERVED_MODEL)
print("HF endpoint       :", HF_ENDPOINT)
print("HF cache (HF_HOME) :", HF_HOME)
print("llama-server      :", LLAMA_SERVER_BIN, "| exists:", LLAMA_SERVER_BIN.exists())
print("API base          :", API_BASE)

Model checkpoint  : unsloth/Qwen3.6-35B-A3B-GGUF:Q4_K_XL
Served model      : user.Qwen3.6-35B-A3B-GGUF
HF endpoint       : http://134.199.133.77
HF cache (HF_HOME) : /workspace/hf_cache
llama-server      : /opt/llama.cpp/build/bin/llama-server | exists: True
API base          : http://127.0.0.1:13305/api/v1


## 2. Prepare Lemonade

Installs `lemonade-sdk` into the current kernel (once), grafts the prebuilt ROCm `llama-server` into
the path Lemonade expects so it **skips the GitHub download**, and registers the model's **Hugging
Face checkpoint** (`repo:variant`) in `user_models.json`. No weights are downloaded here — that
happens in the dedicated download step below. This cell is idempotent.

In [11]:
import json
import subprocess

# 1) Install lemonade-sdk into the current kernel (if not already present).
try:
    import lemonade_server  # noqa: F401
    print("lemonade-sdk: already installed")
except ImportError:
    print("Installing lemonade-sdk ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lemonade-sdk"], check=True)
    print("lemonade-sdk: installed")

os.environ["LEMONADE_CACHE_DIR"] = str(LEMONADE_CACHE)

# 2) Graft the prebuilt ROCm llama.cpp so Lemonade skips its GitHub download.
#    Lemonade looks under <sys.prefix>/rocm/llama_server/build/bin/llama-server.
assert LLAMA_SERVER_BIN.exists(), f"Prebuilt llama-server missing: {LLAMA_SERVER_BIN}"
graft_dir = Path(sys.prefix) / "rocm" / "llama_server"
graft_dir.mkdir(parents=True, exist_ok=True)
build_link = graft_dir / "build"
if not build_link.exists():
    build_link.symlink_to(LLAMA_CPP_DIR / "build")
(graft_dir / "version.txt").write_text(LLAMA_VERSION_ROCM)
(graft_dir / "backend.txt").write_text("rocm")
print("Grafted ROCm llama.cpp:", build_link, "->", build_link.resolve())

# 3) Register the model with its Hugging Face checkpoint (repo:variant).
#    Weights are NOT downloaded here; the pull step below fetches them from the
#    mirror into HF_HOME.
LEMONADE_CACHE.mkdir(parents=True, exist_ok=True)
user_models = {
    MODEL_NAME: {
        "checkpoint": MODEL_CHECKPOINT,
        "recipe": "llamacpp",
        "suggested": True,
        "labels": ["custom", "reasoning"],
        "source": "hf",
    }
}
(LEMONADE_CACHE / "user_models.json").write_text(json.dumps(user_models, indent=2))
print("Registered model:", SERVED_MODEL, "->", MODEL_CHECKPOINT)

lemonade-sdk: already installed
Grafted ROCm llama.cpp: /opt/venv/rocm/llama_server/build -> /opt/llama.cpp/build
Registered model: user.Qwen3.6-35B-A3B-GGUF -> unsloth/Qwen3.6-35B-A3B-GGUF:Q4_K_XL


## 3. Start the Lemonade Server

Starts `lemonade-server-dev serve --llamacpp rocm` on GPU 5 and waits for `/api/v1/health`. If a
server is already running on the port, it is reused instead of started again.

In [12]:
import time
import requests

def server_healthy():
    try:
        return requests.get(f"{API_BASE}/health", timeout=2).ok
    except requests.RequestException:
        return False

if server_healthy():
    print("Lemonade Server already running - reusing:", API_BASE)
else:
    lemonade_bin = Path(sys.executable).parent / "lemonade-server-dev"
    env = os.environ.copy()
    env["HIP_VISIBLE_DEVICES"] = GPU_ID
    env["LEMONADE_CACHE_DIR"] = str(LEMONADE_CACHE)
    env["HF_ENDPOINT"] = HF_ENDPOINT      # download via the mirror
    env["HF_HOME"] = str(HF_HOME)          # cache inside the workspace

    cmd = [
        str(lemonade_bin), "serve",
        "--llamacpp", "rocm",
        "--host", HOST,
        "--port", str(PORT),
        "--ctx-size", str(CTX_SIZE),
    ]
    print("Starting:", " ".join(cmd))
    log_file = SERVER_LOG.open("w")
    proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT,
                            env=env, cwd=str(WORKSPACE))

    for _ in range(60):
        time.sleep(2)
        if proc.poll() is not None:
            raise RuntimeError(f"Server exited early. See {SERVER_LOG}")
        if server_healthy():
            break
    else:
        raise RuntimeError(f"Server not ready in time. See {SERVER_LOG}")
    print(f"Lemonade Server ready (PID {proc.pid}):", API_BASE)

Starting: /opt/venv/bin/lemonade-server-dev serve --llamacpp rocm --host 0.0.0.0 --port 13305 --ctx-size 8192
Lemonade Server ready (PID 446): http://127.0.0.1:13305/api/v1


## 4. Download the model (Lemonade pull)

Downloads the GGUF from the Hugging Face mirror into `HF_HOME` (`/workspace/hf_cache`) via Lemonade's
`/api/v1/pull`. This is the "download to local, then use" step: about **22 GB on first run**, then
cached — subsequent runs return immediately because the file is already local. The `timeout` is set
high to allow the full download.

In [13]:
# Download (pull) the model with a live progress bar.
# The /pull request blocks until finished, so we run it in a background thread and
# poll the HF cache directory size on the main thread to render progress.
import threading
import time

# Expected size of the target GGUF (from the mirror), for the progress bar.
try:
    _tree = requests.get(
        f"{HF_ENDPOINT}/api/models/{MODEL_REPO}/tree/main?recursive=true", timeout=30
    ).json()
    EXPECTED_BYTES = next(
        (e.get("size") or (e.get("lfs") or {}).get("size") or 0)
        for e in _tree
        if e.get("path", "").endswith(f"{MODEL_VARIANT}.gguf")
    )
except Exception as e:
    print("Could not fetch expected size, progress will show downloaded GB only:", e)
    EXPECTED_BYTES = 0

REPO_CACHE = HF_HOME / "hub" / ("models--" + MODEL_REPO.replace("/", "--"))

def _downloaded_bytes():
    """Sum sizes of blob files (including .incomplete) for this repo."""
    total = 0
    if REPO_CACHE.exists():
        for f in REPO_CACHE.rglob("*"):
            if f.is_file():
                try:
                    total += f.stat().st_size
                except OSError:
                    pass
    return total

# Fire the pull in a background thread.
_result = {}
def _do_pull():
    try:
        r = requests.post(f"{API_BASE}/pull", json={"model_name": SERVED_MODEL}, timeout=7200)
        _result["status"] = r.status_code
        _result["json"] = r.json()
    except Exception as ex:
        _result["error"] = repr(ex)

print(f"Pulling {SERVED_MODEL} ({MODEL_CHECKPOINT}) into {HF_HOME} ...")
t = threading.Thread(target=_do_pull, daemon=True)
t.start()

# Poll and render progress until the pull thread finishes.
GB = 1024 ** 3
start = time.time()
last_bytes, last_t = 0, start
while t.is_alive():
    time.sleep(3)
    cur = _downloaded_bytes()
    now = time.time()
    speed = (cur - last_bytes) / max(now - last_t, 1e-6) / 1e6  # MB/s
    last_bytes, last_t = cur, now
    if EXPECTED_BYTES:
        pct = min(cur / EXPECTED_BYTES * 100, 100)
        bar = "#" * int(pct // 4) + "-" * (25 - int(pct // 4))
        eta = (EXPECTED_BYTES - cur) / max(speed * 1e6, 1) if speed > 0 else 0
        print(f"\r[{bar}] {pct:5.1f}%  {cur/GB:5.2f}/{EXPECTED_BYTES/GB:.2f} GB  "
              f"{speed:6.1f} MB/s  ETA {eta/60:4.1f} min", end="", flush=True)
    else:
        print(f"\rDownloaded {cur/GB:5.2f} GB  {speed:6.1f} MB/s", end="", flush=True)

t.join()
print()  # newline after the bar

# Report result.
if _result.get("error"):
    raise RuntimeError(f"Pull failed: {_result['error']}")
print("Pull:", _result.get("json"), "| HTTP", _result.get("status"))

# Confirm the model now shows up as available.
models = requests.get(f"{API_BASE}/models", timeout=30).json()
ids = [m.get("id") for m in models.get("data", [])]
print("Model available:", SERVED_MODEL in ids)


Pulling user.Qwen3.6-35B-A3B-GGUF (unsloth/Qwen3.6-35B-A3B-GGUF:Q4_K_XL) into /workspace/hf_cache ...
[#########################] 100.0%  41.65/20.82 GB  14903.1 MB/s  ETA -0.0 min
Pull: None | HTTP 200
Model available: True


## 5. Load the model and send a request

Loads the now-downloaded model onto GPU 5, then sends an OpenAI-compatible chat request.
`Qwen3.6-35B-A3B` is a **reasoning (thinking) model**, so the chain of thought comes back in
`reasoning_content` and the final answer in `content`. Keep `max_tokens` high enough (e.g. 1024) so
the answer is not cut off by the thinking phase.

In [14]:
# 1) Load the model onto GPU 5 via Lemonade.
r = requests.post(f"{API_BASE}/load", json={"model_name": SERVED_MODEL}, timeout=600)
r.raise_for_status()
print("Load:", r.json())

# 2) OpenAI-compatible chat request.
payload = {
    "model": SERVED_MODEL,
    "messages": [
        {"role": "user", "content": "用一句话中文自我介绍，并说明你运行在什么推理框架上。"},
    ],
    "max_tokens": 1024,
    "temperature": 0.6,
}
resp = requests.post(f"{API_BASE}/chat/completions", json=payload, timeout=300)
resp.raise_for_status()
data = resp.json()
msg = data["choices"][0]["message"]

print("\n--- reasoning_content (first 600 chars) ---")
print((msg.get("reasoning_content") or "").strip()[:600])
print("\n--- answer ---")
print((msg.get("content") or "").strip())
print("\nfinish_reason:", data["choices"][0].get("finish_reason"))
print("tokens/sec   :", round(data.get("timings", {}).get("predicted_per_second", 0), 1))

Load: {'status': 'success', 'message': 'Loaded model: user.Qwen3.6-35B-A3B-GGUF'}

--- reasoning_content (first 600 chars) ---
Here's a thinking process:

1.  **Analyze User Input:**
   - **Requirement 1:** 用一句话中文自我介绍 (Introduce yourself in one sentence in Chinese)
   - **Requirement 2:** 并说明你运行在什么推理框架上 (And state what inference framework you run on)
   - **Language:** Chinese

2.  **Identify Key Constraints & Facts:**
   - I am Qwen (通义千问), developed by Alibaba Group's Tongyi Lab.
   - I need to keep the introduction to exactly one sentence in Chinese.
   - I need to mention the inference framework I run on. As an AI model, I don't "run on" a specific third-party inference framework in the way a user might run a loca

--- answer ---


finish_reason: length
tokens/sec   : 82.5


## 6. (Optional) Measure idle vs. peak VRAM

Samples GPU VRAM with `rocm-smi` while running one long request. llama.cpp preallocates the KV
cache at load time, so the peak is only slightly above the idle (loaded) footprint.

In [ ]:
import threading

def gpu5_used_bytes():
    out = subprocess.run(["rocm-smi", "--showmeminfo", "vram", "-d", GPU_ID],
                         capture_output=True, text=True).stdout
    for line in out.splitlines():
        if "Used Memory" in line:
            return int(line.split()[-1])
    return 0

samples, stop = [], False
def sampler():
    while not stop:
        b = gpu5_used_bytes()
        if b:
            samples.append(b)
        time.sleep(0.2)

t = threading.Thread(target=sampler)
t.start()
try:
    requests.post(f"{API_BASE}/chat/completions", json={
        "model": SERVED_MODEL,
        "messages": [{"role": "user",
                      "content": "详细分析异构加速平台上 AI 与 HPC 协同优化的挑战。" * 40}],
        "max_tokens": 1200,
        "temperature": 0.6,
    }, timeout=300)
finally:
    stop = True
    t.join()

if samples:
    print(f"Idle : {min(samples)/1e9:.2f} GB")
    print(f"Peak : {max(samples)/1e9:.2f} GB")
    print(f"Delta: {(max(samples)-min(samples))/1e9:.2f} GB")
else:
    print("No VRAM samples captured.")

Idle : 22.64 GB
Peak : 22.95 GB
Delta: 0.31 GB


## 7. Stop the Lemonade Server

Stops the server started by this notebook and frees GPU 5.

In [ ]:
# Stop the Lemonade Server and its llama-server child (frees GPU 5).
subprocess.run(["pkill", "-f", "lemonade-server-dev"], check=False)
subprocess.run(["pkill", "-f", "llama-server"], check=False)
print("Stopped Lemonade Server and llama-server processes.")